Imports

In [13]:
# ── Imports ─────────────────────────────────────────────────────────────────────
import json
import pandas as pd
import numpy as np
from collections import Counter

In [14]:
# ── Scales ─────────────────────────────────────────────────────────────────────
 
SCALE5  = list(range(1, 6))
SCALE10 = list(range(1, 11))

In [15]:
# ── DFU Utilities ─────────────────────────────────────────────────────────────
 
def dfu(input_data, histogram_input=True, normalised=True):
    """The Distance From Unimodality measure.
    :param input_data: the data, by default the relative frequencies of ratings
    :param histogram_input: False to compute rel. frequencies (ratings as input)
    :param normalised: whether to normalise the output
    :return: the DFU score
    """
    hist      = input_data if histogram_input else to_hist(input_data)
    max_value = max(hist)
    pos_max   = np.where(hist == max_value)[0][0]
    max_diff  = 0
    for i in range(pos_max, len(hist)-1):
        diff = hist[i+1] - hist[i]
        if diff > max_diff:
            max_diff = diff
    for i in range(pos_max, 0, -1):
        diff = hist[i-1] - hist[i]
        if diff > max_diff:
            max_diff = diff
    if normalised:
        return max_diff / max_value
    return max_diff
 
 
def to_hist(scores, bins_num=3, normed=True):
    """Create a normalised histogram.
    :param scores: the ratings (not necessarily discrete)
    :param bins_num: the number of bins to create
    :param normed: whether to normalise or not
    :return: the histogram
    """
    counts, bins  = np.histogram(a=scores, bins=bins_num)
    counts_normed = counts / counts.sum()
    return counts_normed if normed else counts
 
 
def pdf(scores, scale=SCALE10):
    """Relative frequencies of ordinal ratings.
    :param scores: the ratings
    :param scale: the rating categories, by default 10-point scale
    :return: the relative frequencies
    """
    freqs = Counter(scores)
    return np.array([freqs[s] / len(scores) for s in scale])
 
 
def cpdf(scores, scale=SCALE10):
    """Cumulative relative frequencies.
    :param scores: the ratings
    :param scale: the rating categories, by default 10-point scale
    :return: the cumulative relative frequencies
    """
    return np.cumsum(pdf(scores, scale))
 
 
def ndfu(ratings, scale):
    """Normalised DFU for a list of raw ratings over a fixed scale.
    :param ratings: list of raw ratings
    :param scale: rating scale (e.g. range(1,6))
    :return: nDFU score in [0, 1]
    """
    return dfu(pdf(ratings, scale=scale), histogram_input=True, normalised=True)

In [16]:
# ── RPA ────────────────────────────────────────────────────────────────────────
 
def rpa(ratings, subgroups, scale, variant='max'):
    """Compute Residual Polarization Attribution (RPA) for a dimension split.
 
    All variants are bounded in [0, 1] since nDFU ∈ [0, 1].
 
    Variants:
        'max' (Strict)           — reduction based on worst-case subgroup nDFU.
                                   Primary splitting criterion. Generalises AU.
        'min' (Lenient)          — reduction based on best-case subgroup nDFU.
                                   Detects asymmetric / one-sided polarization.
        'var' (Variance-weighted)— size-weighted reduction across all subgroups.
                                   Analogous to Ward's method in clustering.
        'avg' (Average)          — unweighted mean reduction across subgroups.
                                   Equals 'var' when subgroups are balanced.
 
    :param ratings:   full annotation list for the text
    :param subgroups: list of annotation lists, one per subgroup
    :param scale:     rating scale (e.g. range(1,6))
    :param variant:   one of 'max', 'min', 'var', 'avg'
    :return:          RPA score (float)
    """
    ndfu_global = ndfu(ratings, scale)
    ndfu_subs   = [ndfu(g, scale) for g in subgroups]
    N           = len(ratings)
 
    if variant == 'max':
        return abs(ndfu_global - max(ndfu_subs))
    elif variant == 'min':
        return abs(ndfu_global - min(ndfu_subs))
    elif variant == 'var':
        return abs(ndfu_global - sum((len(g)/N) * s
                                     for g, s in zip(subgroups, ndfu_subs)))
    elif variant == 'avg':
        return abs(ndfu_global - sum(ndfu_subs) / len(ndfu_subs))
    else:
        raise ValueError(
            f"Unknown variant '{variant}'. Choose from 'max', 'min', 'var', 'avg'.")

In [17]:
# ── Top-k Filtering ────────────────────────────────────────────────────────────
 
def top_k_polarized(texts_ratings, scale, k):
    """Return the top-k most polarized texts by nDFU score.
 
    :param texts_ratings: list of rating lists, one per text
    :param scale:         rating scale (e.g. range(1,6))
    :param k:             number of texts to return
    :return:              list of (index, ratings, ndfu_score) sorted descending
    """
    if k <= 0:
        raise ValueError(f"k must be a positive integer, got {k}.")
 
    n = len(texts_ratings)
    if k > n:
        print(f"Warning: requested k={k} exceeds dataset size "
              f"({n} texts). Returning all {n} texts.")
        k = n
 
    scored = [
        (i, ratings, ndfu(ratings, scale))
        for i, ratings in enumerate(texts_ratings)
    ]
    return sorted(scored, key=lambda x: x[2], reverse=True)[:k]

In [18]:
# ── Preprocessing ──────────────────────────────────────────────────────────────
 
def preprocess_dataset(df, rating_col='ratings', exclude_cols=None):
    """Convert a DataFrame into the annotation format expected by PolarizedTree.
 
    INPUT FORMAT (pandas DataFrame)
    --------------------------------
    Each row represents one text. Columns are:
        - 'id'       : unique identifier for the text
        - 'text'     : the raw text content
        - rating_col : a list of ratings, one per annotator
                       e.g. [2, 2, 3, 5, 4, 1]
        - dim cols   : one column per socio-demographic dimension,
                       each containing a list of values aligned with ratings
                       e.g. 'gender'   -> ['man', 'man', 'woman', 'woman', 'man', 'woman']
                            'politics' -> ['left', 'right', 'left', 'right', 'center', 'left']
                            'age'      -> ['young', 'young', 'old', 'old', 'young', 'old']
 
    OUTPUT FORMAT (list of instance dicts)
    ----------------------------------------
    Each instance dict contains:
        - 'id'          : the text id
        - 'text'        : the raw text
        - 'annotations' : list of (rating, attrs) tuples, one per annotator
                          e.g. [(2, {'gender': 'man', 'politics': 'left', 'age': 'young'}),
                                (2, {'gender': 'man', 'politics': 'right', 'age': 'young'}),
                                (3, {'gender': 'woman', 'politics': 'left', 'age': 'old'}),
                                ...]
 
    Also returns the set of dimension names (dims) automatically extracted
    from the DataFrame columns, so you do not need to specify them manually.
 
    :param df:           pandas DataFrame with one row per text
    :param rating_col:   name of the column containing rating lists
    :param exclude_cols: columns to exclude from annotator attributes
                         defaults to ['id', 'text', rating_col]
    :return:             (dataset, dims)
                         dataset : list of instance dicts
                         dims    : set of dimension names
    """
    if exclude_cols is None:
        exclude_cols = ['id', 'text', rating_col]
 
    dim_cols = [c for c in df.columns if c not in exclude_cols]
    dataset  = []
 
    for _, row in df.iterrows():
        ratings     = row[rating_col]
        annotations = [
            (rating, {dim: row[dim][i] for dim in dim_cols})
            for i, rating in enumerate(ratings)
        ]
        dataset.append({
            'id':          row['id'],
            'text':        row['text'],
            'annotations': annotations
        })
 
    return dataset, set(dim_cols)

In [19]:
# ── PolarizedTree ──────────────────────────────────────────────────────────────
 
class PolarizedTree:
    """Polarized Tree for detecting and attributing annotation polarization
    to socio-demographic or ideological dimensions via recursive DFS splitting.
 
    Each node is a list of (rating, annotator_attributes) tuples.
    The tree is built by selecting at each node the dimension that maximises
    RPA_max, subject to stopping criteria (min_size, h, max_depth, dim exhaustion).
    Leaf nodes are labelled with a pole (toxic / civil / indeterminate) based
    on the proportion of toxic ratings p_{i,g}.
 
    Parameters
    ----------
    scale              : rating scale, e.g. range(1, 6)
    min_size           : minimum subgroup size for a valid split
    h                  : minimum RPA_max gain to accept a split (lower threshold)
    max_depth          : maximum tree depth (user-defined hard limit)
    rpa_variant        : RPA variant used for splitting ('max','min','var','avg')
    toxicity_threshold : ratings >= this value are considered toxic
    """
 
    def __init__(self, scale, min_size=5, h=0.05, max_depth=5,
                 rpa_variant='max', toxicity_threshold=3):
        self.scale              = scale
        self.min_size           = min_size
        self.h                  = h
        self.max_depth          = max_depth
        self.rpa_variant        = rpa_variant
        self.toxicity_threshold = toxicity_threshold
 
    # ── Internal helpers ───────────────────────────────────────────────────────
 
    def _groups(self, dim, node):
        """Partition node annotations by values of dimension dim.
        Annotators missing a value for dim are silently skipped.
        :param dim:  dimension name (str)
        :param node: list of (rating, annotator_attributes) tuples
        :return:     dict of {dim_value: [(rating, attrs), ...]}
        """
        groups = {}
        for rating, attrs in node:
            val = attrs.get(dim)
            if val is None:
                continue
            if val not in groups:
                groups[val] = []
            groups[val].append((rating, attrs))
        return groups
 
    def _node_ratings(self, node):
        """Extract raw ratings from a node (discards annotator attributes)."""
        return [r for r, _ in node]
 
    def _make_leaf(self, node, path, node_ndfu):
        """Create a leaf dict."""
        return {
            'path':        path,
            'annotations': node,
            'ndfu':        node_ndfu
        }
 
    def _compute_pole(self, leaf):
        """Compute pole label for a leaf using p_{i,g}.
        p_{i,g} = proportion of toxic ratings in the subgroup.
        Returns: (pole, p) where pole is 'toxic' | 'civil' | 'indeterminate'
        """
        ratings = self._node_ratings(leaf['annotations'])
        p = sum(1 for r in ratings if r >= self.toxicity_threshold) / len(ratings)
        if p > 0.5:   return 'toxic', p
        elif p < 0.5: return 'civil', p
        else:         return 'indeterminate', p
 
    # ── DFS ───────────────────────────────────────────────────────────────────
 
    def _dfs(self, node, dims, depth, path, verbose):
        """Recursive depth-first splitting.
        Selects the dimension maximising RPA_max at each node.
        Stops when any stopping criterion is met.
        """
        node_ndfu    = ndfu(self._node_ratings(node), self.scale)
        node_ratings = self._node_ratings(node)
 
        if verbose:
            path_str = ' → '.join(f"{d}={v}" for d, v in path) or 'ROOT'
            print(f"\n{'  ' * (depth-1)}[Depth {depth}] Node: {path_str}")
            print(f"{'  ' * (depth-1)}  nDFU={node_ndfu:.3f} | n={len(node)}")
 
        # ── Stopping criteria ──────────────────────────────────────────────
        if depth > self.max_depth:
            if verbose:
                print(f"{'  ' * (depth-1)}  ✋ LEAF: max_depth reached ({self.max_depth})")
            return [self._make_leaf(node, path, node_ndfu)]
 
        if len(dims) == 0:
            if verbose:
                print(f"{'  ' * (depth-1)}  ✋ LEAF: no dimensions remaining")
            return [self._make_leaf(node, path, node_ndfu)]
 
        # ── Find best split by RPA_max ─────────────────────────────────────
        best_dim       = None
        best_rpa_score = 0
        rpa_scores     = {}
 
        for dim in dims:
            groups           = self._groups(dim, node)
            subgroup_ratings = [self._node_ratings(g) for g in groups.values()]
 
            if not all(len(g) >= self.min_size for g in subgroup_ratings):
                rpa_scores[dim] = None
                continue
 
            rpa_score       = rpa(node_ratings, subgroup_ratings,
                                  self.scale, variant=self.rpa_variant)
            rpa_scores[dim] = rpa_score
 
            if rpa_score > best_rpa_score:
                best_rpa_score = rpa_score
                best_dim       = dim
 
        if verbose:
            print(f"{'  ' * (depth-1)}  RPA scores per dimension:")
            for dim, score in sorted(rpa_scores.items()):
                if score is None:
                    print(f"{'  ' * (depth-1)}    {dim}: SKIPPED (min_size not met)")
                else:
                    marker = ' ◀ BEST' if dim == best_dim else ''
                    print(f"{'  ' * (depth-1)}    {dim}: {score:.3f}{marker}")
 
        if best_dim is None or best_rpa_score <= self.h:
            reason = "no valid dimension" if best_dim is None \
                     else f"RPA={best_rpa_score:.3f} ≤ h={self.h}"
            if verbose:
                print(f"{'  ' * (depth-1)}  ✋ LEAF: {reason}")
            return [self._make_leaf(node, path, node_ndfu)]
 
        if verbose:
            print(f"{'  ' * (depth-1)}  ✂ SPLIT on '{best_dim}' "
                  f"(RPA={best_rpa_score:.3f})")
 
        # ── Recurse into children ──────────────────────────────────────────
        remaining_dims = dims - {best_dim}
        leaves         = []
 
        for val, child in self._groups(best_dim, node).items():
            child_path = path + [(best_dim, val)]
            leaves.extend(
                self._dfs(child, remaining_dims, depth + 1, child_path, verbose))
 
        return leaves
 
    # ── Public API ─────────────────────────────────────────────────────────────
 
    def fit(self, annotations, dims, verbose=False):
        """Fit the polarized tree on a single text's annotations.
        :param annotations: list of (rating, annotator_attributes) tuples
        :param dims:        set of candidate dimensions
        :param verbose:     if True, print step-by-step debug info
        :return:            self
        """
        if len(annotations) < self.min_size:
            raise ValueError(
                f"Not enough annotations ({len(annotations)}) "
                f"for min_size={self.min_size}.")
 
        if verbose:
            global_ndfu = ndfu(self._node_ratings(annotations), self.scale)
            print("=" * 60)
            print("FITTING POLARIZED TREE")
            print(f"  Total annotations : {len(annotations)}")
            print(f"  Global nDFU       : {global_ndfu:.3f}")
            print(f"  Dimensions        : {dims}")
            print(f"  min_size={self.min_size} | h={self.h} | "
                  f"max_depth={self.max_depth} | variant={self.rpa_variant}")
            print("=" * 60)
 
        self.leaves_ = self._dfs(
            annotations, set(dims), depth=1, path=[], verbose=verbose)
        return self
 
    def get_poles(self):
        """Compute pole labels for all leaf nodes.
        :return: list of dicts with keys: path, n, ndfu, p_toxic, pole
        """
        if not hasattr(self, 'leaves_'):
            raise RuntimeError("Call fit() before get_poles().")
        results = []
        for leaf in self.leaves_:
            pole, p_value = self._compute_pole(leaf)
            results.append({
                'path':    leaf['path'],
                'n':       len(leaf['annotations']),
                'ndfu':    round(leaf['ndfu'], 3),
                'p_toxic': round(p_value, 3),
                'pole':    pole
            })
        return results
 
    def find_polyphony(self):
        """Find leaf nodes with differing majority opinions (Stage 2).
        Indeterminate leaves are excluded via Contextual Pruning.
        :return: dict {pole_label: [leaf_nodes]} if polyphony exists, else {}
        """
        if not hasattr(self, 'leaves_'):
            raise RuntimeError("Call fit() before find_polyphony().")
        majority_groups = {}
        for leaf in self.leaves_:
            pole, _ = self._compute_pole(leaf)
            if pole == 'indeterminate':
                continue
            if pole not in majority_groups:
                majority_groups[pole] = []
            majority_groups[pole].append(leaf)
        return majority_groups if len(majority_groups) > 1 else {}
 
    def print_tree(self):
        """Print a text-based visualization of the polarized tree."""
        if not hasattr(self, 'leaves_'):
            raise RuntimeError("Call fit() before print_tree().")
 
        global_ratings = []
        for leaf in self.leaves_:
            global_ratings.extend(self._node_ratings(leaf['annotations']))
        global_ndfu = ndfu(global_ratings, self.scale)
 
        print("\n" + "=" * 60)
        print("POLARIZED TREE")
        print("=" * 60)
        print(f"ROOT | nDFU={global_ndfu:.3f} | n={len(global_ratings)}")
        print("|")
 
        for i, leaf in enumerate(self.leaves_):
            pole, p     = self._compute_pole(leaf)
            path        = leaf['path']
            is_last     = (i == len(self.leaves_) - 1)
            pole_symbol = ("🔴" if pole == 'toxic'
                           else "🔵" if pole == 'civil'
                           else "⚪")
 
            for j, (dim, val) in enumerate(path):
                indent       = "    " * j
                is_leaf_step = (j == len(path) - 1)
                if is_leaf_step:
                    conn = "└──" if is_last else "├──"
                    print(f"{indent}{conn} {dim}={val}")
                    print(f"{indent}    nDFU={leaf['ndfu']:.3f} | "
                          f"n={len(leaf['annotations'])} | "
                          f"p_toxic={p:.2f} | {pole_symbol} {pole}")
                else:
                    print(f"{indent}└── {dim}={val}")
 
            if i < len(self.leaves_) - 1:
                print("|")
 
        print("=" * 60)
 
    def to_dict(self, instance_id, text):
        """Serialize tree results to a JSON-compatible dict for saving.
        :param instance_id: the text id
        :param text:        the raw text
        :return:            dict with all results
        """
        if not hasattr(self, 'leaves_'):
            raise RuntimeError("Call fit() before to_dict().")
 
        poles     = self.get_poles()
        polyphony = self.find_polyphony()
 
        global_ratings = []
        for leaf in self.leaves_:
            global_ratings.extend(self._node_ratings(leaf['annotations']))
 
        return {
            'id':           instance_id,
            'text':         text,
            'global_ndfu':  round(ndfu(global_ratings, self.scale), 3),
            'n_annotators': len(global_ratings),
            'polyphony':    len(polyphony) > 1,
            'leaves': [
                {
                    'path':    p['path'],
                    'n':       p['n'],
                    'ndfu':    p['ndfu'],
                    'p_toxic': p['p_toxic'],
                    'pole':    p['pole']
                }
                for p in poles
            ]
        }
 

In [20]:
# ── Dataset Runner ─────────────────────────────────────────────────────────────
 
def run_dataset(dataset, dims, scale, save_path, min_size=5, h=0.05,
                max_depth=5, rpa_variant='max', toxicity_threshold=3):
    """Run the PolarizedTree on every instance in the dataset and save results.
 
    :param dataset:            list of instance dicts from preprocess_dataset()
    :param dims:               set of dimension names
    :param scale:              rating scale (e.g. range(1,6))
    :param save_path:          path to save results as JSON
    :param min_size:           minimum subgroup size
    :param h:                  minimum RPA gain threshold (lower threshold)
    :param max_depth:          maximum tree depth
    :param rpa_variant:        RPA variant for splitting
    :param toxicity_threshold: ratings >= this are toxic
    """
    tree = PolarizedTree(
        scale=scale, min_size=min_size, h=h,
        max_depth=max_depth, rpa_variant=rpa_variant,
        toxicity_threshold=toxicity_threshold
    )
 
    results = []
    skipped = []
 
    for instance in dataset:
        try:
            tree.fit(instance['annotations'], dims,verbose=True)
            results.append(tree.to_dict(instance['id'], instance['text']))
        except ValueError as e:
            skipped.append({'id': instance['id'], 'reason': str(e)})
            print(f"Skipped instance {instance['id']}: {e}")
 
    output = {
        'params': {
            'min_size':           min_size,
            'h':                  h,
            'max_depth':          max_depth,
            'rpa_variant':        rpa_variant,
            'toxicity_threshold': toxicity_threshold,
            'dims':               list(dims)
        },
        'n_processed': len(results),
        'n_skipped':   len(skipped),
        'skipped':     skipped,
        'results':     results
    }
 
    with open(save_path, 'w') as f:
        json.dump(output, f, indent=2)
 
    print(f"\n✅ Done. Processed={len(results)} | Skipped={len(skipped)}")
    print(f"   Saved to: {save_path}")

In [21]:
# ── Visualize Single Instance ──────────────────────────────────────────────────
 
def visualize_instance(save_path, instance_id):
    """Load saved results and visualize a single instance.
 
    :param save_path:   path to the JSON file saved by run_dataset()
    :param instance_id: the id of the instance to visualize
    """
    with open(save_path, 'r') as f:
        output = json.load(f)
 
    instance = next(
        (r for r in output['results'] if r['id'] == instance_id), None)
 
    if instance is None:
        skipped = next(
            (s for s in output['skipped'] if s['id'] == instance_id), None)
        if skipped:
            print(f"Instance {instance_id} was skipped: {skipped['reason']}")
        else:
            print(f"Instance {instance_id} not found in results.")
        return
 
    # ── Header ────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"INSTANCE {instance_id}")
    print("=" * 60)
    print(f"Text        : {instance['text']}")
    print(f"Annotators  : {instance['n_annotators']}")
    print(f"Global nDFU : {instance['global_ndfu']}")
    print(f"Polyphony   : {'✅ YES' if instance['polyphony'] else '❌ NO'}")
 
    # ── Tree ──────────────────────────────────────────────────────────────────
    print(f"\nROOT | nDFU={instance['global_ndfu']} | n={instance['n_annotators']}")
    print("|")
 
    leaves = instance['leaves']
    for i, leaf in enumerate(leaves):
        path        = leaf['path']
        is_last     = (i == len(leaves) - 1)
        pole        = leaf['pole']
        pole_symbol = ("🔴" if pole == 'toxic'
                       else "🔵" if pole == 'civil'
                       else "⚪")
 
        for j, (dim, val) in enumerate(path):
            indent       = "    " * j
            is_leaf_step = (j == len(path) - 1)
            if is_leaf_step:
                conn = "└──" if is_last else "├──"
                print(f"{indent}{conn} {dim}={val}")
                print(f"{indent}    nDFU={leaf['ndfu']} | "
                      f"n={leaf['n']} | "
                      f"p_toxic={leaf['p_toxic']} | "
                      f"{pole_symbol} {pole}")
            else:
                print(f"{indent}└── {dim}={val}")
 
        if i < len(leaves) - 1:
            print("|")
 
    # ── Poles summary ──────────────────────────────────────────────────────────
    print("\n" + "-" * 60)
    print("POLES SUMMARY")
    print("-" * 60)
    for leaf in leaves:
        path_str    = ' → '.join(f"{d}={v}" for d, v in leaf['path']) or 'root'
        pole_symbol = ("🔴" if leaf['pole'] == 'toxic'
                       else "🔵" if leaf['pole'] == 'civil'
                       else "⚪")
        print(f"  {pole_symbol} {leaf['pole']:>14} | {path_str}")
 
    print("=" * 60)

In [22]:
# ── Example usage ──────────────────────────────────────────────────────────────
 
if __name__ == '__main__':
 
    data = {
        'id':   [0, 1],
        'text': ["misogynistic comment", "racist comment"],
        'ratings': [
            [1, 2, 1, 2, 1, 5, 4, 5, 4, 5, 1, 2, 1, 5, 4, 5],
            [1, 1, 2, 5, 5, 4, 3, 3, 2, 1, 4, 5, 4, 5, 3, 3],
        ],
        'gender': [
            ['female','female','female','female','female',
             'male','male','male','male','male',
             'male','male','male','male','male','male'],
            ['female','female','female','male','male',
             'female','male','male','female','female',
             'male','male','female','male','female','male'],
        ],
        'orientation': [
            ['heterosexual','heterosexual','lgbtq+','heterosexual','lgbtq+',
             'heterosexual','heterosexual','heterosexual','heterosexual','heterosexual',
             'lgbtq+','lgbtq+','lgbtq+','lgbtq+','lgbtq+','lgbtq+'],
            ['heterosexual','lgbtq+','heterosexual','heterosexual','lgbtq+',
             'heterosexual','heterosexual','lgbtq+','lgbtq+','heterosexual',
             'heterosexual','lgbtq+','heterosexual','lgbtq+','heterosexual','heterosexual'],
        ],
        'politics': [
            ['left','right','left','center','right',
             'right','right','center','left','right',
             'left','left','left','right','right','right'],
            ['left','left','right','right','left',
             'center','right','left','right','left',
             'right','right','left','left','right','center'],
        ],
    }
 
    df = pd.DataFrame(data)
 
    # Preprocess
    dataset, dims = preprocess_dataset(df)
    print(f"Dimensions detected : {dims}")
    print(f"Instances           : {len(dataset)}")
 
    run_dataset(
        dataset=dataset,
        dims=dims,
        scale=SCALE5,
        save_path='results.json',  # ← updated
        min_size=3,
        h=0.05,
        max_depth=5,
        rpa_variant='max'
    )

    visualize_instance('results.json', instance_id=0)  # ← updated
    visualize_instance('results.json', instance_id=1)  # ← updated

Dimensions detected : {'gender', 'politics', 'orientation'}
Instances           : 2
FITTING POLARIZED TREE
  Total annotations : 16
  Global nDFU       : 0.600
  Dimensions        : {'gender', 'politics', 'orientation'}
  min_size=3 | h=0.05 | max_depth=5 | variant=max

[Depth 1] Node: ROOT
  nDFU=0.600 | n=16
  RPA scores per dimension:
    gender: 0.400 ◀ BEST
    orientation: 0.067
    politics: SKIPPED (min_size not met)
  ✂ SPLIT on 'gender' (RPA=0.400)

  [Depth 2] Node: gender=female
    nDFU=0.000 | n=5
    RPA scores per dimension:
      orientation: SKIPPED (min_size not met)
      politics: SKIPPED (min_size not met)
    ✋ LEAF: no valid dimension

  [Depth 2] Node: gender=male
    nDFU=0.200 | n=11
    RPA scores per dimension:
      orientation: 0.300 ◀ BEST
      politics: SKIPPED (min_size not met)
    ✂ SPLIT on 'orientation' (RPA=0.300)

    [Depth 3] Node: gender=male → orientation=heterosexual
      nDFU=0.000 | n=5
      RPA scores per dimension:
        politics: S